# AIV-Guard: Orchid-HEval Classification Evaluation

This notebook evaluates a frozen single-agent classification pipeline using Qwen2.5-3B-Instruct. The evaluation set contains 250 requirements derived from 50 Orchid-HEval source tasks, with one clear case and four ambiguity variants per task.

**Dataset:** `SII-YDD/Orchid`, `Orchid-HEval/data.jsonl`. The selected CSV stored in the project's `dataset/` directory. If no local copy is available, the original selection procedure reconstructs the set from the published source.

**Execution environment:** Python with the dependencies listed in the project. A CUDA GPU is required for model inference. Google Colab supports GPU execution. To store checkpoints in Google Drive, set `AIV_GUARD_MOUNT_DRIVE=1` before running the notebook.

**Outputs:** Dataset manifest, protocol, classification records, summary metrics, and label-conflict audit. CEV is evaluated separately in `02_CEV_Review30.ipynb`.


In [ ]:
# Dependencies for notebook execution
%pip -q install 'transformers>=4.45,<5' 'accelerate>=0.33,<2' safetensors 'huggingface_hub>=0.25,<2' 'pandas>=2' 'scikit-learn>=1.3'


In [ ]:
# Resolve the project dataset and the experiment output directory.
from pathlib import Path
import os, json, hashlib, random
import pandas as pd

try:
    from google.colab import drive
except ImportError:
    drive = None

# Drive storage is optional. Set AIV_GUARD_MOUNT_DRIVE=1 for persistent Colab checkpoints.
if drive is not None and os.environ.get('AIV_GUARD_MOUNT_DRIVE', '0') == '1':
    drive.mount('/content/drive')

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/AIV-Guard-GitHub'),
    Path('/content/drive/MyDrive/AIV-Guard-GitHub'),
]
PROJECT_ROOT = next(
    (p for p in PROJECT_CANDIDATES
     if (p / 'dataset' / 'orchid_new250_cases.csv').is_file()),
    None,
)

DEFAULT_OUTPUT = (
    Path('/content/drive/MyDrive/AIV_Guard_New250_SingleAgent_Frozen')
    if Path('/content/drive/MyDrive').exists()
    else (PROJECT_ROOT or Path.cwd()) / 'outputs'
)
WORK = Path(os.environ.get('AIV_GUARD_OUTPUT_DIR', str(DEFAULT_OUTPUT)))
WORK.mkdir(parents=True, exist_ok=True)
print('Project dataset:', str(PROJECT_ROOT / 'dataset') if PROJECT_ROOT else 'rebuild/cache mode')
print('Results directory:', WORK)


In [ ]:
# Load the packaged evaluation set when available; otherwise reconstruct it from Orchid.
from huggingface_hub import hf_hub_download
import json, hashlib, random, shutil
import pandas as pd

DATASET_ID = 'SII-YDD/Orchid'
DATASET_FILE = 'Orchid-HEval/data.jsonl'
SOURCE_FILE = WORK / 'orchid_heval_official_source.jsonl'
SELECTION_SEED = 42
OLD_125_TASKS=['HumanEval/0', 'HumanEval/1', 'HumanEval/117', 'HumanEval/118', 'HumanEval/130', 'HumanEval/135', 'HumanEval/143', 'HumanEval/149', 'HumanEval/39', 'HumanEval/49', 'HumanEval/5', 'HumanEval/50', 'HumanEval/54', 'HumanEval/58', 'HumanEval/6', 'HumanEval/62', 'HumanEval/75', 'HumanEval/8', 'HumanEval/80', 'HumanEval/81', 'HumanEval/82', 'HumanEval/86', 'HumanEval/90', 'HumanEval/93', 'HumanEval/96']
LABELS=[
    ('clear','prompt',''),
    ('lexical','Lexical_prompt','Lexical_ambiguity_explanation'),
    ('semantic','Semantic_prompt','Semantic_ambiguity_explanation'),
    ('syntactic','Syntactic_prompt','Syntactic_ambiguity_explanation'),
    ('vagueness','Vagueness_prompt','Vagueness_ambiguity_explanation'),
]


PROTOCOL_FILE = WORK / 'protocol.json'
RECORDED_PROTOCOL = (
    json.loads(PROTOCOL_FILE.read_text(encoding='utf-8'))
    if PROTOCOL_FILE.exists() else {}
)
LOCAL_DATA = (
    PROJECT_ROOT / 'dataset' / 'orchid_new250_cases.csv'
    if PROJECT_ROOT is not None else None
)
CACHED_DATA = WORK / 'orchid_new250_cases.csv'

if LOCAL_DATA is not None:
    DATAFILE = LOCAL_DATA
    if CACHED_DATA.is_file():
        assert hashlib.sha256(DATAFILE.read_bytes()).digest() == hashlib.sha256(CACHED_DATA.read_bytes()).digest(), (
            'Packaged dataset differs from the original experiment dataset.'
        )
elif CACHED_DATA.is_file():
    DATAFILE = CACHED_DATA
else:
    # The original deterministic selection is retained for dataset reconstruction.
    if not SOURCE_FILE.is_file():
        downloaded = hf_hub_download(
            repo_id=DATASET_ID, repo_type='dataset', filename=DATASET_FILE
        )
        shutil.copyfile(downloaded, SOURCE_FILE)
    rawbytes = SOURCE_FILE.read_bytes()
    source_rows = [json.loads(line) for line in rawbytes.decode('utf-8-sig').splitlines() if line.strip()]
    assert len(source_rows) == 164, 'Unexpected number of Orchid-HEval source tasks'
    assert len({str(r['name']) for r in source_rows}) == 164, 'Repeated source task IDs'
    required_fields = {'name','entry_point'} | {key for _, key, _ in LABELS}
    assert all(required_fields.issubset(r) for r in source_rows), 'Orchid dataset schema changed'

    available = sorted(
        [r for r in source_rows if str(r['name']) not in OLD_125_TASKS],
        key=lambda r: int(str(r['name']).split('/')[-1]),
    )
    assert len(available) == 139, 'Unexpected number of candidate tasks'
    sampled_names = sorted(
        random.Random(SELECTION_SEED).sample([str(r['name']) for r in available], 50),
        key=lambda s: int(s.split('/')[-1]),
    )
    selected = set(sampled_names)
    new_cases = []
    for source in source_rows:
        if str(source['name']) not in selected:
            continue
        for label, key, explain_key in LABELS:
            req = source[key]
            assert isinstance(req, str) and req.strip(), (source['name'], label)
            new_cases.append({
                'task_name': str(source['name']),
                'entry_point': str(source['entry_point']),
                'requirement': req,
                'gold_label': label,
                'gold_binary': 'CLEAR' if label == 'clear' else 'AMBIGUOUS',
                'gold_explanation': str(source.get(explain_key, '') or ''),
            })
    new_cases = sorted(
        new_cases,
        key=lambda x: (
            int(x['task_name'].split('/')[-1]),
            [lab for lab, _, _ in LABELS].index(x['gold_label']),
        ),
    )
    for i, case in enumerate(new_cases, 1):
        case['case_id'] = i
    df_reconstructed = pd.DataFrame(new_cases)[
        ['case_id','task_name','entry_point','requirement',
         'gold_label','gold_binary','gold_explanation']
    ]
    DATAFILE = CACHED_DATA
    df_reconstructed.to_csv(DATAFILE, index=False)

# Validate the complete selected dataset against the fixed evaluation protocol.
df = pd.read_csv(DATAFILE, keep_default_na=False)
required_columns = {
    'case_id','task_name','entry_point','requirement',
    'gold_label','gold_binary','gold_explanation',
}
assert required_columns.issubset(df.columns), 'Dataset columns do not match the protocol'
df = df[['case_id','task_name','entry_point','requirement',
         'gold_label','gold_binary','gold_explanation']].copy()
df['case_id'] = pd.to_numeric(df['case_id'], errors='raise').astype(int)
assert len(df) == 250 and list(df.case_id) == list(range(1, 251)), 'Invalid case count or ID ordering'
assert df.case_id.is_unique and df.task_name.nunique() == 50
assert df.groupby('task_name').size().eq(5).all(), 'Every source task requires five cases'
assert df.gold_label.value_counts().to_dict() == {label: 50 for label, _, _ in LABELS}
assert df.gold_binary.value_counts().to_dict() == {'CLEAR': 50, 'AMBIGUOUS': 200}
assert df.requirement.map(lambda x: isinstance(x, str) and bool(x.strip())).all()
assert (df.gold_binary == df.gold_label.map(lambda x: 'CLEAR' if x == 'clear' else 'AMBIGUOUS')).all()
assert set(df.task_name).isdisjoint(OLD_125_TASKS), 'Overlap with prior source tasks'

expected_names = sorted(
    random.Random(SELECTION_SEED).sample(
        [f'HumanEval/{i}' for i in range(164) if f'HumanEval/{i}' not in OLD_125_TASKS], 50
    ),
    key=lambda s: int(s.split('/')[-1]),
)
selected_tasks = sorted(df.task_name.unique().tolist(), key=lambda s: int(s.split('/')[-1]))
assert selected_tasks == expected_names, 'Source-task selection differs from seed 42'
expected_rows = sorted(
    [(t, label) for t in selected_tasks for label, _, _ in LABELS],
    key=lambda pair: (int(pair[0].split('/')[-1]),
                      [label for label, _, _ in LABELS].index(pair[1])),
)
assert list(zip(df.task_name, df.gold_label)) == expected_rows, 'Case ordering differs from protocol'

if PROJECT_ROOT is not None:
    manifest_path = PROJECT_ROOT / 'dataset' / 'manifest.csv'
    if manifest_path.is_file():
        reference_manifest = pd.read_csv(manifest_path, keep_default_na=False)
        cols = ['case_id','task_name','gold_label','gold_binary']
        assert df[cols].astype(str).reset_index(drop=True).equals(
            reference_manifest[cols].astype(str).reset_index(drop=True)
        ), 'Packaged manifest does not match the dataset'

DATA_SHA = hashlib.sha256(DATAFILE.read_bytes()).hexdigest()
if RECORDED_PROTOCOL.get('selected_dataset_sha256') is not None:
    assert DATA_SHA == RECORDED_PROTOCOL['selected_dataset_sha256'], 'Dataset hash differs from recorded run'
SOURCE_SHA = (
    hashlib.sha256(SOURCE_FILE.read_bytes()).hexdigest()
    if SOURCE_FILE.is_file()
    else RECORDED_PROTOCOL.get('source_sha256')
)
if SOURCE_FILE.is_file() and RECORDED_PROTOCOL.get('source_sha256'):
    assert SOURCE_SHA == RECORDED_PROTOCOL['source_sha256'], 'Source hash differs from recorded run'

canonical = df.requirement.astype(str).str.split().str.join(' ')
audit = df.assign(canonical_requirement=canonical)
label_counts = audit.groupby('canonical_requirement').gold_binary.nunique()
conflicting_texts = set(label_counts[label_counts > 1].index)
audit[audit.canonical_requirement.isin(conflicting_texts)].to_csv(
    WORK / 'label_conflicts_audit.csv', index=False
)
df[['case_id','task_name','gold_label','gold_binary']].to_csv(
    WORK / 'manifest.csv', index=False
)
print('Cases:', len(df), '| Source tasks:', len(selected_tasks))
print('Selected dataset SHA-256:', DATA_SHA)
print('Source SHA-256:', SOURCE_SHA if SOURCE_SHA is not None else 'not available')
print('Conflicting normalized texts:', len(conflicting_texts))


In [ ]:
# Frozen model and classification prompt used in the reported experiment.
import hashlib,json
MODEL_ID='Qwen/Qwen2.5-3B-Instruct'
SEED=42
MAX_NEW_TOKENS=130
AGENT1_SYSTEM = """You are Agent 1, a software-requirements ambiguity classifier.
Judge the wording of the requirement itself, not what a programmer might prefer.
CLEAR means no reasonable behavioral ambiguity is supported by the wording.
AMBIGUOUS means at least two defensible readings change program behavior for some input.
Do not invent a reading that contradicts an explicit constraint or example.
Do not assume vague terms are precise unless the requirement defines them.
Return ONLY compact JSON with keys: label, evidence, reason.
label is exactly CLEAR or AMBIGUOUS. For AMBIGUOUS, evidence must quote an EXACT substring of the requirement.
For CLEAR, evidence must be an empty string. reason must be brief. No markdown."""
PROMPT_SHA=hashlib.sha256(AGENT1_SYSTEM.encode('utf-8')).hexdigest()
PROTOCOL={
  'experiment':'AIV_Guard_Orchid_HEval_250_new_source_tasks_frozen',
  'dataset_source':'SII-YDD/Orchid Orchid-HEval train Orchid-HEval/data.jsonl',
  'source_sha256':SOURCE_SHA,
  'selected_dataset_sha256':DATA_SHA,
  'selection_seed':SELECTION_SEED,
  'old_source_tasks_excluded':OLD_125_TASKS,
  'new_source_task_count':50,
  'new_cases_total':250,
  'evaluation_split':'ALL 250 are new, source-task-disjoint evaluation, NO new dev split',
  'source_task_ids':selected_tasks,
  'model_id':MODEL_ID,
  'generation':'greedy (do_sample=False), explicit attention_mask, one call per requirement',
  'seed':SEED,'max_new_tokens':MAX_NEW_TOKENS,
  'prompt_sha256':PROMPT_SHA,
  'gold_given_to_model':False,
  'label_parser':'exact CLEAR/AMBIGUOUS, normalize observed DEV typo AMBIGOUS only',
  'gate':'CLEAR with empty evidence OR AMBIGUOUS with whitespace-normalized evidence quote; otherwise NEEDS_REVIEW',
  'evaluation':'report raw Agent 1 labels separately from gate; unknown/abstention counted incorrect; macro-F1',
  'data_limitations':'all 250 from same published Orchid dataset, not external domain validation',
}
PROTOCOL_FILE=WORK/'protocol.json'
if PROTOCOL_FILE.exists():
    assert json.loads(PROTOCOL_FILE.read_text(encoding='utf-8'))==PROTOCOL, 'Existing protocol mismatch! STOP.'
else:
    PROTOCOL_FILE.write_text(json.dumps(PROTOCOL,indent=2,ensure_ascii=False),encoding='utf-8')
print('Fixed Agent 1 prompt SHA:',PROMPT_SHA)
print('Evaluation protocol initialized for 250 cases.')


In [ ]:
# Load the Qwen classification model.
import torch,transformers,time
from transformers import AutoTokenizer,AutoModelForCausalLM
assert torch.cuda.is_available(), 'CUDA GPU required for model inference'
torch.manual_seed(SEED)
print('GPU:',torch.cuda.get_device_name(0),'Transformers:',transformers.__version__)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,
                       torch_dtype=torch.float16,device_map='auto',low_cpu_mem_usage=True)
model.eval()
print('Model loaded:',MODEL_ID)


In [ ]:
# Classification, deterministic evidence checks, and checkpoint handling.
import os,time,json
import pandas as pd

VALID={'CLEAR','AMBIGUOUS'}
def normspace(x):
    return ' '.join(str(x).split())

def parse_agent1(raw,requirement):
    a={'label':'INVALID','raw_label':'','typo_fixed':False,'parsed':False,
       'evidence':'','reason':'','quote_valid':False,'format_issue':''}
    try:
        start=raw.find('{')
        if start<0: raise ValueError('No JSON object')
        data,_=json.JSONDecoder().raw_decode(raw[start:])
        if not isinstance(data,dict):raise ValueError('Response not JSON object')
        raw_label=data.get('label','')
        if not isinstance(raw_label,str):raise ValueError('label not string')
        label='AMBIGUOUS' if raw_label=='AMBIGOUS' else raw_label
        if label not in VALID:raise ValueError('Invalid label '+raw_label[:50])
        evidence=data.get('evidence','')
        if not isinstance(evidence,str):raise ValueError('evidence not string')
        quote=bool(evidence.strip()) and normspace(evidence) in normspace(requirement)
        a.update(label=label,raw_label=raw_label,typo_fixed=(raw_label=='AMBIGOUS'),
                 parsed=True,evidence=evidence,reason=str(data.get('reason','')),
                 quote_valid=quote)
        if label=='CLEAR' and evidence.strip():
            a['format_issue']='CLEAR_with_nonempty_evidence'
        if label=='AMBIGUOUS' and not quote:
            a['format_issue']='AMBIGUOUS_without_matching_quote'
    except Exception as exc:
        a['format_issue']=f'{type(exc).__name__}: {str(exc)[:130]}'
    return a

def gate_action(a):
    if not a['parsed']:return 'NEEDS_REVIEW'
    if a['label']=='CLEAR' and not a['evidence'].strip():return 'CLEAR'
    if a['label']=='AMBIGUOUS' and a['quote_valid']:return 'AMBIGUOUS'
    return 'NEEDS_REVIEW'

# Parser and gate unit checks with synthetic examples.
assert parse_agent1('{"label":"AMBIGOUS","evidence":"quickly"}','respond quickly')['label']=='AMBIGUOUS'
assert parse_agent1('{"label":"AMBIGUOUS","evidence":"hello world"}', 'hello\n world')['quote_valid']
assert not parse_agent1('{"label":"AMBIGUOUS","evidence":"tomorrow"}', 'respond quickly')['quote_valid']
assert gate_action(parse_agent1('{"label":"CLEAR","evidence":""}','do x'))=='CLEAR'
assert gate_action(parse_agent1('{"label":"INVALID"}','do x'))=='NEEDS_REVIEW'

COLUMNS=['case_id','task_name','gold_label','gold_binary','agent1_label','raw_label',
         'typo_fixed','parsed','quote_valid','format_issue','review_action',
         'evidence','reason','raw_response','seconds','status']
PRED_FILE=WORK/'predictions_test250.csv'
def read_checkpoint():
    if not PRED_FILE.exists():return pd.DataFrame(columns=COLUMNS)
    saved=pd.read_csv(PRED_FILE,keep_default_na=False)
    assert set(COLUMNS).issubset(saved.columns), 'Checkpoint schema unexpected'
    assert saved.case_id.is_unique, 'Duplicate case IDs in saved checkpoint'
    assert set(saved.case_id).issubset(set(df.case_id)), 'Case ID outside 250 manifest'
    reference = df.set_index('case_id')
    rows = saved.set_index('case_id')
    for field in ['task_name', 'gold_label', 'gold_binary']:
        assert (rows[field].astype(str) == reference.loc[rows.index, field].astype(str)).all(), (
            f'Checkpoint mismatch in {field}'
        )
    return saved

def save_checkpoint(x):
    temp=WORK/'_predictions_test250_temp.csv'
    x.sort_values('case_id').to_csv(temp,index=False)
    os.replace(str(temp),str(PRED_FILE))

def call_agent1(requirement):
    messages=[{'role':'system','content':AGENT1_SYSTEM},
              {'role':'user','content':'Requirement:\n'+str(requirement)}]
    ids=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
                                      return_tensors='pt').to(model.device)
    mask=torch.ones_like(ids)
    with torch.inference_mode():
        outputs=model.generate(input_ids=ids,attention_mask=mask,
                               max_new_tokens=MAX_NEW_TOKENS,do_sample=False,
                               pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0,ids.shape[1]:],skip_special_tokens=True).strip()

def run_test250():
    saved=read_checkpoint()
    done=set(saved.loc[saved.status=='ok','case_id'].astype(int))
    print(f'Start / Resume: {len(done)}/250 completed; {250-len(done)} remaining')
    for row in df.itertuples(index=False):
        if row.case_id in done:continue
        start=time.time()
        try:
            raw=call_agent1(row.requirement)
            a=parse_agent1(raw,row.requirement)
            result={
                'case_id':int(row.case_id),'task_name':row.task_name,
                'gold_label':row.gold_label,'gold_binary':row.gold_binary,
                'agent1_label':a['label'],'raw_label':a['raw_label'],
                'typo_fixed':a['typo_fixed'],'parsed':a['parsed'],
                'quote_valid':a['quote_valid'],'format_issue':a['format_issue'],
                'review_action':gate_action(a),'evidence':a['evidence'],
                'reason':a['reason'],'raw_response':raw,
                'seconds':round(time.time()-start,2),'status':'ok',
            }
            saved.loc[len(saved)]=[result[k] for k in COLUMNS]
            save_checkpoint(saved)
            done.add(int(row.case_id))
            print(f"case {row.case_id:03d}/250 | Gold={row.gold_binary:9s} | Agent1={a['label']:9s} | Gate={result['review_action']:12s}",flush=True)
        except Exception as exc:
            save_checkpoint(saved)
            print('STOP at case',row.case_id, type(exc).__name__,str(exc)[:200])
            print('Completed predictions are stored in the checkpoint.')
            raise
    print('Completed:',len(done),'/250')
print('Classification and validation functions initialized.')


In [ ]:
# Run classification, resuming from the current checkpoint if available.
run_test250()


In [ ]:
# Compute evaluation metrics from completed prediction records.
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,confusion_matrix
import numpy as np, json
results=read_checkpoint().sort_values('case_id')
assert len(results)==250 and results.status.eq('ok').all(),f'Only {len(results)}/250 cases completed. Finish cell 7 first.'
assert list(results.case_id)==list(df.case_id), 'Checkpoint has missing or extra cases'

CLASSES=['CLEAR','AMBIGUOUS']
def score(ytrue,ypred):
    yt=list(ytrue);yp=list(ypred)
    p,r,f,s=precision_recall_fscore_support(yt,yp,labels=CLASSES,zero_division=0)
    return {'n':len(yt),'accuracy':round(float(accuracy_score(yt,yp)),4),
            'macro_f1':round(float(f.mean()),4),
            'clear_precision':round(float(p[0]),4),
            'clear_recall':round(float(r[0]),4),
            'clear_f1':round(float(f[0]),4),
            'ambiguous_precision':round(float(p[1]),4),
            'ambiguous_recall':round(float(r[1]),4),
            'ambiguous_f1':round(float(f[1]),4),
            'confusion_labels':['CLEAR','AMBIGUOUS','INVALID','NEEDS_REVIEW'],
            'confusion_matrix':confusion_matrix(yt,yp,labels=['CLEAR','AMBIGUOUS','INVALID','NEEDS_REVIEW']).tolist()}

accepted=results[results.review_action.isin(CLASSES)]
summary={
    'experiment':'Single-Agent new250 Orchid-HEval',
    'completed':True,'cases':250,'source_tasks':50,
    'n_clear':int((results.gold_binary=='CLEAR').sum()),
    'n_ambiguous':int((results.gold_binary=='AMBIGUOUS').sum()),
    'source_sha256':SOURCE_SHA,'dataset250_sha256':DATA_SHA,
    'prompt_sha256':PROMPT_SHA,'model_id':MODEL_ID,
    'agent1_raw_classification':score(results.gold_binary,results.agent1_label),
    'gate_all_cases_review_counted_wrong':score(results.gold_binary,results.review_action),
    'gate_accepted_n':int(len(accepted)),
    'gate_coverage':round(len(accepted)/250,4),
    'gate_accepted_accuracy':round(float((accepted.review_action==accepted.gold_binary).mean()),4) if len(accepted) else None,
    'needs_review':int((results.review_action=='NEEDS_REVIEW').sum()),
    'format_errors':int(results.format_issue.astype(str).str.len().gt(0).sum()),
    'typo_normalizations':int(results.typo_fixed.astype(str).str.lower().eq('true').sum()),
    'always_ambiguous_dummy_not_a_run':score(results.gold_binary,['AMBIGUOUS']*250),
    'per_gold_ambiguity_type':{
         label:{'count':len(group),
                'agent1_correct':int((group.agent1_label==group.gold_binary).sum()),
                'agent1_accuracy':round(float((group.agent1_label==group.gold_binary).mean()),4)}
         for label,group in results.groupby('gold_label')},
    'scope_note':'This is Orchid-HEval only, with 50 source tasks never used in prior 125-case experiment. Not external-domain validation.',
    'gate_limit':'The Validation Gate tests JSON and quote consistency, not actual semantics or execution.',
}
(SUMMARY_FILE:=WORK/'summary_test250.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
print('AIV-Guard Agent1 Accuracy:',summary['agent1_raw_classification']['accuracy'])
print('AIV-Guard Agent1 Macro-F1:',summary['agent1_raw_classification']['macro_f1'])
print('Always ambiguous dummy:',summary['always_ambiguous_dummy_not_a_run']['accuracy'],
      summary['always_ambiguous_dummy_not_a_run']['macro_f1'])
print('Gate review:',summary['needs_review'],'Accepted:',summary['gate_accepted_n'])
print('Confusion matrix rows Actual CLEAR,AMBIGUOUS; cols Pred CLEAR,AMBIGUOUS,INVALID,NEEDS_REVIEW:')
print(np.array(summary['agent1_raw_classification']['confusion_matrix'])[:2,:])
print('Metric files in:',WORK)


In [ ]:
# Package the dataset and reproducibility artifacts with the evaluation results.
from pathlib import Path
import zipfile
try:
    from google.colab import files
except ImportError:
    files = None

bundle = WORK / 'AIV_Guard_NEW250_SingleAgent_Results.zip'
artifacts = {
    'orchid_new250_cases.csv': DATAFILE,  # The selected dataset in dataset/ (or reconstructed copy).
    'manifest.csv': WORK / 'manifest.csv',
    'protocol.json': WORK / 'protocol.json',
    'predictions_test250.csv': WORK / 'predictions_test250.csv',
    'summary_test250.json': WORK / 'summary_test250.json',
    'label_conflicts_audit.csv': WORK / 'label_conflicts_audit.csv',
}
missing = [name for name, path in artifacts.items() if not Path(path).is_file()]
assert not missing, f'Missing artifacts: {missing}. Run the preceding cells first.'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name, path in artifacts.items():
        archive.write(path, arcname=name)
print('Results archive:', bundle)
print('Included:', ', '.join(artifacts))
if files is not None:
    files.download(str(bundle))
